# Notebook 2 — Deep Learning Forecasting Pipeline
## Industrial IoT Telemetry | 5-Second Device Readings | 3-Hour Horizon

> **Continuation notebook — Deep Learning Track.**  
> Run after `EDA_and_arima.ipynb` (or execute Cell 0 below to bootstrap from disk).  
> Assumes `df_5s`, `train`, `test`, and shared constants are available.

---
## Should We Even Use Deep Learning Here?

Before writing a single line of PyTorch, we must honestly answer this question.

| Signal Property | Implication for DL |
|---|---|
| Strong short-lag autocorrelation (ACF ≫ 0 at lags 1–100) | ✅ LSTM/GRU can exploit sequential memory effectively |
| ~500k observations, 5-second granularity | ✅ Enough data to train sequence models without overfitting |
| Structural outages / zero-runs | ⚠️ Sequences spanning outages need masking or regime tokens |
| Weak long-range seasonality | ✅ No need for Transformer global attention at day-level periods |
| 2160-step forecast horizon | ⚠️ Recursive DL will accumulate error; direct/seq2seq preferred |
| Single univariate target | ⚠️ DL advantage shrinks vs. well-engineered ML on univariate series |

**Verdict:** Deep learning is *justified but not obviously superior* to the ML pipeline.  
The most likely winner is a **Temporal CNN** (fast, parallelisable, no vanishing gradient)  
or a **seq2seq GRU** (natural multi-step decoder).  
We implement and compare all architectures so the results speak for themselves.


## Cell 0 — Environment Bootstrap

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                     'grid.alpha': 0.3, 'grid.linestyle': '--'})

# ── Constants ─────────────────────────────────────────────────────────────────
FREQ_SEC         = 5
HOURLY           = int(3600 / FREQ_SEC)       # 720
DAILY            = int(86400 / FREQ_SEC)      # 17 280
FORECAST_HORIZON = int(3 * 3600 / FREQ_SEC)  # 2 160
TEST_DAYS        = 7

if 'df_5s' not in dir():
    df = pd.read_excel("problem_data.xlsx")
    df = df.dropna(subset=["Reading"])
    df = df.drop_duplicates(subset=["timestamp"], keep="first")
    df = df.sort_values("timestamp")
    df_5s = (
        df.set_index("timestamp")[["Reading"]]
        .resample("5s").sum()
        .reset_index()
    )
    df_5s["date"] = df_5s["timestamp"].dt.date
    print("Reloaded df_5s from disk.")
else:
    print("df_5s already in namespace.")

if 'train' not in dir():
    split_index = int(len(df_5s) - TEST_DAYS * DAILY)
    train = df_5s.iloc[:split_index].copy()
    test  = df_5s.iloc[split_index:].copy()

GLOBAL_MEAN = float(train["Reading"].mean())
GLOBAL_STD  = float(train["Reading"].std())
print(f"Train: {train.shape}  |  Test: {test.shape}")
print(f"Global mean: {GLOBAL_MEAN:.4f}  |  std: {GLOBAL_STD:.4f}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("  No GPU found — using CPU. Training will be slower.")
    print("  Tip: Enable GPU runtime in Colab / Kaggle for 5-10× speedup.")


---
# Stage 1 — Data Preparation for Deep Learning

## Normalisation Strategy

IoT sensor signals often have non-stationary scale.  
We use **Z-score normalisation fit on training data only** — no leakage.  
All DL models train on normalised values; predictions are inverse-transformed  
back to original scale before evaluation.

## Sequence Generation & Sliding Window

Each training sample is a tuple `(X_window, Y_horizon)`:

```
X_window : [t - LOOKBACK + 1 … t]   shape: (LOOKBACK,  n_features)
Y_horizon: [t+1 … t+HORIZON]        shape: (HORIZON,)
```

This is the **MIMO strategy** — one forward pass produces all 2160 targets.  
It eliminates recursive error accumulation at the cost of a larger output head.

**LOOKBACK choice:** We use `LOOKBACK = 2160` (= horizon) so the model sees  
exactly as far back as it must predict forward — a principled symmetric window.  
For memory-constrained environments, `LOOKBACK = 720` (1 hour) is a safe fallback.


In [ ]:
# ── Normalisation ─────────────────────────────────────────────────────────────
series_all = df_5s["Reading"].values.astype(np.float32)
train_vals = train["Reading"].values.astype(np.float32)
test_vals  = test["Reading"].values.astype(np.float32)

# Fit scaler on train only
MU    = float(train_vals.mean())
SIGMA = float(train_vals.std()) + 1e-8

def normalise(x):  return (x - MU) / SIGMA
def denormalise(x): return x * SIGMA + MU

train_norm = normalise(train_vals)
test_norm  = normalise(test_vals)

# Full series for context (train + test concatenated after normalisation)
full_norm = normalise(series_all)

print(f"Normalised train — mean: {train_norm.mean():.4f}  std: {train_norm.std():.4f}")
print(f"Normalised test  — mean: {test_norm.mean():.4f}  std: {test_norm.std():.4f}")


In [ ]:
# ── Sliding Window Dataset ─────────────────────────────────────────────────────
LOOKBACK = 720    # 1 hour of context at 5-second resolution
# Use LOOKBACK=2160 if GPU memory permits — broader context, better long-range

class SlidingWindowDataset(Dataset):
    """
    Generates (X, Y) pairs via sliding window over a 1D time series.

    Parameters
    ----------
    series   : 1D np.array of normalised values
    lookback : number of past steps used as input
    horizon  : number of future steps to predict (MIMO output)
    stride   : step between consecutive windows (>1 reduces dataset size)
    """
    def __init__(self, series, lookback, horizon, stride=1):
        self.series   = torch.tensor(series, dtype=torch.float32)
        self.lookback = lookback
        self.horizon  = horizon
        self.stride   = stride
        n = len(series) - lookback - horizon + 1
        self.indices  = list(range(0, n, stride))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        x = self.series[start : start + self.lookback]       # (lookback,)
        y = self.series[start + self.lookback :
                        start + self.lookback + self.horizon] # (horizon,)
        return x.unsqueeze(-1), y   # x: (L, 1), y: (H,)


# Validation split: last 20% of training series
val_size  = int(0.2 * len(train_norm))
tr_series = train_norm[:-val_size]
va_series = train_norm[-(val_size + LOOKBACK):]   # overlap for lookback context

STRIDE     = 5     # every 5 steps = 25-second stride → manageable dataset size
BATCH_SIZE = 64

train_ds = SlidingWindowDataset(tr_series, LOOKBACK, FORECAST_HORIZON, stride=STRIDE)
val_ds   = SlidingWindowDataset(va_series, LOOKBACK, FORECAST_HORIZON, stride=STRIDE)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=0, pin_memory=(DEVICE.type=="cuda"))
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=0, pin_memory=(DEVICE.type=="cuda"))

print(f"Lookback     : {LOOKBACK} steps ({LOOKBACK*5/60:.1f} min)")
print(f"Horizon      : {FORECAST_HORIZON} steps ({FORECAST_HORIZON*5/3600:.1f} hr)")
print(f"Train windows: {len(train_ds):,}  |  Val windows: {len(val_ds):,}")
print(f"Batches/epoch: {len(train_dl):,}")

# Quick shape check
xb, yb = next(iter(train_dl))
print(f"\nBatch shapes — X: {tuple(xb.shape)}  Y: {tuple(yb.shape)}")


---
# Stage 2 — Shared Training Infrastructure

Defining reusable training loop, callbacks, and evaluation utilities  
shared by all DL architectures below.


In [ ]:
# ── Training utilities ────────────────────────────────────────────────────────

class EarlyStopping:
    """Stops training when validation loss stops improving."""
    def __init__(self, patience=7, min_delta=1e-5):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_loss  = np.inf
        self.counter    = 0
        self.best_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.counter    = 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore_best(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


def train_model(model, train_dl, val_dl, n_epochs=50, lr=1e-3,
                patience=10, model_name="Model"):
    """
    Standard training loop with:
      - AdamW optimiser + cosine annealing LR schedule
      - Early stopping on validation MAE
      - Loss curve tracking
    Returns: (train_losses, val_losses)
    """
    model = model.to(DEVICE)
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs, eta_min=lr*0.01)
    loss_fn = nn.HuberLoss(delta=1.0)   # robust to spikes / outage boundaries

    es = EarlyStopping(patience=patience)

    train_losses, val_losses = [], []

    for epoch in range(1, n_epochs + 1):
        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        ep_loss = 0.0
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            pred = model(xb)         # (B, H)
            loss = loss_fn(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            ep_loss += loss.item()
        ep_loss /= len(train_dl)

        # ── Validate ──────────────────────────────────────────────────────────
        model.eval()
        vl = 0.0
        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                pred = model(xb)
                vl  += loss_fn(pred, yb).item()
        vl /= len(val_dl)

        train_losses.append(ep_loss)
        val_losses.append(vl)
        sched.step()

        if epoch % 5 == 0 or epoch == 1:
            print(f"  [{model_name}] Epoch {epoch:3d}/{n_epochs}  "
                  f"Train: {ep_loss:.5f}  Val: {vl:.5f}  "
                  f"LR: {sched.get_last_lr()[0]:.2e}")

        if es.step(vl, model):
            print(f"  Early stopping at epoch {epoch}. Best val: {es.best_loss:.5f}")
            es.restore_best(model)
            break

    return train_losses, val_losses


def dl_recursive_forecast(model, seed_series_norm, n_steps=FORECAST_HORIZON):
    """
    MIMO inference: pass the last LOOKBACK steps → get H-step forecast.
    If H < n_steps, repeat with rolling window (chunked MIMO).
    """
    model.eval()
    predictions = []
    buffer = list(seed_series_norm[-LOOKBACK:])

    steps_done = 0
    while steps_done < n_steps:
        x = torch.tensor(buffer[-LOOKBACK:], dtype=torch.float32)
        x = x.unsqueeze(0).unsqueeze(-1).to(DEVICE)   # (1, L, 1)
        with torch.no_grad():
            pred = model(x).squeeze(0).cpu().numpy()  # (H,)

        chunk = pred[:n_steps - steps_done]
        predictions.extend(chunk.tolist())
        buffer.extend(chunk.tolist())
        steps_done += len(chunk)

    return denormalise(np.array(predictions[:n_steps], dtype=np.float32))


def plot_loss_curve(train_losses, val_losses, model_name):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(train_losses, label="Train Loss (Huber)")
    ax.plot(val_losses,   label="Val Loss (Huber)")
    ax.set_title(f"{model_name} — Training Curve")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Huber Loss")
    ax.legend()
    plt.tight_layout()
    plt.show()


def evaluate_dl(y_true, y_pred, model_name, y_train_raw=None):
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    yt, yp = y_true[mask], y_pred[mask]
    mae  = float(mean_absolute_error(yt, yp))
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    denom = (np.abs(yt) + np.abs(yp)) / 2
    m = denom > 0
    smape = float(np.mean(np.abs(yt[m]-yp[m])/denom[m])*100)
    result = dict(Model=model_name, MAE=mae, RMSE=rmse, SMAPE=smape)
    print(f"\n{'─'*40}\n  {model_name}\n{'─'*40}")
    print(f"  MAE   : {mae:.4f}\n  RMSE  : {rmse:.4f}\n  SMAPE : {smape:.2f}%")
    return result


def plot_dl_forecast(y_true, y_pred, model_name, n_show=2160):
    n = min(n_show, len(y_true), len(y_pred))
    fig, axes = plt.subplots(2, 1, figsize=(14, 7),
                             gridspec_kw={"height_ratios": [3, 1]})
    axes[0].plot(y_true[:n],  label="Actual",    linewidth=0.8, alpha=0.8)
    axes[0].plot(y_pred[:n],  label="Predicted", linewidth=0.9, alpha=0.85, color="tomato")
    axes[0].set_title(f"{model_name} — Actual vs Predicted", fontsize=12)
    axes[0].legend(); axes[0].set_ylabel("Reading")
    residuals = np.array(y_true[:n]) - np.array(y_pred[:n])
    axes[1].bar(range(n), residuals, alpha=0.4, color="steelblue", width=1)
    axes[1].axhline(0, color="black", linewidth=0.8)
    axes[1].set_title("Residuals"); axes[1].set_xlabel("Step")
    plt.tight_layout(); plt.show()


dl_leaderboard = []   # global registry for this notebook
print("DL training utilities ready.")


---
# Stage 3 — Model A: Temporal CNN (TCN)

## Why TCN First?

> *"For most sequence modelling benchmarks, CNNs outperform RNNs."*  
> — Bai et al., 2018 ("An Empirical Evaluation of Generic Convolutional and Recurrent Networks")

**Key properties:**  
- **Causal convolutions** — no future leakage  
- **Dilated convolutions** — exponentially growing receptive field without depth  
  (`dilation = 2^i` at layer `i` → layer 8 sees 256 steps back)  
- **Residual connections** — stable gradient flow, avoids vanishing gradient  
- **Fully parallelisable** — much faster to train than RNNs on GPU  
- **Deterministic at inference** — no hidden state to initialise  

**Suitability for this problem:**  
- Strong short-range ACF → shallow dilations capture it perfectly  
- No need for global attention (weak long seasonality) → simpler than Transformer  
- Large dataset → parallelism advantage over LSTM is significant


In [ ]:
class CausalConv1d(nn.Module):
    """Causal (left-padded) 1D convolution — no future leakage."""
    def __init__(self, in_ch, out_ch, kernel_size, dilation=1):
        super().__init__()
        self.pad  = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size,
                              dilation=dilation, padding=0)

    def forward(self, x):
        # x: (B, C, L)
        x = nn.functional.pad(x, (self.pad, 0))
        return self.conv(x)


class TCNBlock(nn.Module):
    """
    One TCN residual block:
      CausalConv → LayerNorm → GELU → Dropout →
      CausalConv → LayerNorm → GELU → Dropout +
      residual (1x1 projection if channels differ)
    """
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.1):
        super().__init__()
        self.conv1 = CausalConv1d(in_ch,  out_ch, kernel_size, dilation)
        self.conv2 = CausalConv1d(out_ch, out_ch, kernel_size, dilation)
        self.norm1 = nn.LayerNorm(out_ch)
        self.norm2 = nn.LayerNorm(out_ch)
        self.act   = nn.GELU()
        self.drop  = nn.Dropout(dropout)
        self.proj  = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        # x: (B, C, L)
        res = self.proj(x)
        out = self.drop(self.act(self.norm1(self.conv1(x).transpose(1,2)).transpose(1,2)))
        out = self.drop(self.act(self.norm2(self.conv2(out).transpose(1,2)).transpose(1,2)))
        return out + res


class TCNForecaster(nn.Module):
    """
    Temporal CNN for multi-step (MIMO) time-series forecasting.

    Architecture:
      Input projection → N TCN blocks (exponential dilation) →
      Global average pool → FC head → FORECAST_HORIZON outputs
    """
    def __init__(self, lookback, horizon, n_channels=64, n_layers=6,
                 kernel_size=3, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Conv1d(1, n_channels, 1)

        blocks = []
        for i in range(n_layers):
            dilation = 2 ** i    # 1, 2, 4, 8, 16, 32
            blocks.append(TCNBlock(n_channels, n_channels, kernel_size,
                                   dilation, dropout))
        self.tcn_blocks = nn.Sequential(*blocks)

        self.head = nn.Sequential(
            nn.Linear(n_channels, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, horizon),
        )

    def forward(self, x):
        # x: (B, L, 1) → permute to (B, 1, L)
        x = x.permute(0, 2, 1)
        x = self.input_proj(x)         # (B, C, L)
        x = self.tcn_blocks(x)         # (B, C, L)
        x = x.mean(dim=-1)             # global average pool: (B, C)
        return self.head(x)            # (B, H)


# ── Instantiate & inspect ─────────────────────────────────────────────────────
tcn = TCNForecaster(
    lookback   = LOOKBACK,
    horizon    = FORECAST_HORIZON,
    n_channels = 64,
    n_layers   = 6,
    kernel_size= 3,
    dropout    = 0.1,
)
tcn = tcn.to(DEVICE)

total_params = sum(p.numel() for p in tcn.parameters() if p.requires_grad)
print(f"TCN — Trainable parameters: {total_params:,}")

# Receptive field of the TCN stack
rf = sum([(3-1) * (2**i) for i in range(6)]) + 1
print(f"TCN receptive field: {rf} steps = {rf * 5 / 60:.1f} minutes")


In [ ]:
print("Training TCN...")
tcn_train_losses, tcn_val_losses = train_model(
    tcn, train_dl, val_dl,
    n_epochs   = 50,
    lr         = 1e-3,
    patience   = 10,
    model_name = "TCN",
)
plot_loss_curve(tcn_train_losses, tcn_val_losses, "TCN")


In [ ]:
# ── Inference ─────────────────────────────────────────────────────────────────
seed = train_norm   # use all training data as seed context
tcn_preds = dl_recursive_forecast(tcn, seed, n_steps=FORECAST_HORIZON)

y_test_raw = test["Reading"].values[:FORECAST_HORIZON]
tcn_result = evaluate_dl(y_test_raw, tcn_preds, "TCN (MIMO 3-hr)")
dl_leaderboard.append(tcn_result)
plot_dl_forecast(y_test_raw, tcn_preds, "TCN")


---
# Stage 4 — Model B: GRU Encoder–Decoder (Seq2Seq)

## Why Seq2Seq GRU?

The standard approach for multi-step forecasting with RNNs is the  
**encoder–decoder** architecture:

```
Encoder GRU:  processes input window → produces context vector h
Decoder GRU:  conditioned on h, autoregressively generates H predictions
```

**Suitability:**  
- Encoder compresses temporal history into a fixed-size state — ideal for  
  capturing the "regime" at each point (normal vs outage)  
- Decoder's hidden state naturally propagates sequential uncertainty  
- `teacher_forcing_ratio` during training improves convergence  
- GRU is preferred over LSTM here: fewer parameters, similar performance  
  on sequences < 1000 steps, and faster iteration

**Tradeoff vs TCN:**  
- GRU trains slower (sequential hidden state, non-parallelisable along time axis)  
- But GRU may capture outage recovery dynamics better through its reset gate  
- On GPU the speed gap is smaller; on CPU TCN is significantly faster


In [ ]:
class GRUEncoderDecoder(nn.Module):
    """
    Seq2Seq GRU for MIMO time-series forecasting.

    Encoder: stacked GRU processes input window → final hidden state
    Decoder: single GRU step repeated H times, conditioned on encoder state
    """
    def __init__(self, input_size=1, hidden_size=128, num_layers=2,
                 horizon=FORECAST_HORIZON, dropout=0.2):
        super().__init__()
        self.horizon     = horizon
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        # ── Encoder ───────────────────────────────────────────────────────────
        self.encoder = nn.GRU(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0,
        )

        # ── Decoder ───────────────────────────────────────────────────────────
        self.decoder = nn.GRU(
            input_size  = 1,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0,
        )

        self.output_proj = nn.Linear(hidden_size, 1)
        self.drop        = nn.Dropout(dropout)

    def forward(self, x, teacher_forcing_ratio=0.0, targets=None):
        """
        x       : (B, L, 1) — encoder input
        targets : (B, H)    — ground truth for teacher forcing (train only)
        """
        B = x.size(0)

        # ── Encode ────────────────────────────────────────────────────────────
        _, h = self.encoder(x)           # h: (num_layers, B, hidden)

        # ── Decode ────────────────────────────────────────────────────────────
        dec_input = x[:, -1:, :]         # last observed value: (B, 1, 1)
        outputs   = []

        for t in range(self.horizon):
            out, h = self.decoder(dec_input, h)   # out: (B, 1, hidden)
            pred   = self.output_proj(self.drop(out))  # (B, 1, 1)
            outputs.append(pred.squeeze(-1))            # (B, 1)

            # Teacher forcing: feed ground truth vs. own prediction
            if targets is not None and np.random.rand() < teacher_forcing_ratio:
                dec_input = targets[:, t].unsqueeze(1).unsqueeze(-1)
            else:
                dec_input = pred

        return torch.cat(outputs, dim=1)   # (B, H)


# ── Teacher-forcing training wrapper ─────────────────────────────────────────
def train_seq2seq(model, train_dl, val_dl, n_epochs=50, lr=1e-3,
                  patience=10, model_name="GRU Seq2Seq"):
    """Like train_model but supports teacher forcing for seq2seq GRUs."""
    model = model.to(DEVICE)
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs, eta_min=lr*0.01)
    loss_fn = nn.HuberLoss(delta=1.0)
    es = EarlyStopping(patience=patience)
    train_losses, val_losses = [], []

    for epoch in range(1, n_epochs + 1):
        # Teacher forcing anneals from 0.5 → 0.0 over first half of training
        tf_ratio = max(0.0, 0.5 * (1 - epoch / (n_epochs * 0.5)))

        model.train()
        ep_loss = 0.0
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            pred = model(xb, teacher_forcing_ratio=tf_ratio, targets=yb)
            loss = loss_fn(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_loss += loss.item()
        ep_loss /= len(train_dl)

        model.eval()
        vl = 0.0
        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                pred = model(xb, teacher_forcing_ratio=0.0)
                vl  += loss_fn(pred, yb).item()
        vl /= len(val_dl)

        train_losses.append(ep_loss)
        val_losses.append(vl)
        sched.step()

        if epoch % 5 == 0 or epoch == 1:
            print(f"  [{model_name}] Epoch {epoch:3d}/{n_epochs}  "
                  f"Train: {ep_loss:.5f}  Val: {vl:.5f}  TF: {tf_ratio:.2f}")

        if es.step(vl, model):
            print(f"  Early stopping at epoch {epoch}.")
            es.restore_best(model)
            break

    return train_losses, val_losses


gru_seq2seq = GRUEncoderDecoder(
    hidden_size = 128,
    num_layers  = 2,
    horizon     = FORECAST_HORIZON,
    dropout     = 0.2,
)
total_params = sum(p.numel() for p in gru_seq2seq.parameters() if p.requires_grad)
print(f"GRU Seq2Seq — Trainable parameters: {total_params:,}")


In [ ]:
print("Training GRU Encoder-Decoder...")
gru_train_losses, gru_val_losses = train_seq2seq(
    gru_seq2seq, train_dl, val_dl,
    n_epochs   = 50,
    lr         = 5e-4,
    patience   = 10,
    model_name = "GRU Seq2Seq",
)
plot_loss_curve(gru_train_losses, gru_val_losses, "GRU Encoder-Decoder")


In [ ]:
gru_preds  = dl_recursive_forecast(gru_seq2seq, train_norm, n_steps=FORECAST_HORIZON)
gru_result = evaluate_dl(y_test_raw, gru_preds, "GRU Seq2Seq (MIMO 3-hr)")
dl_leaderboard.append(gru_result)
plot_dl_forecast(y_test_raw, gru_preds, "GRU Encoder-Decoder")


---
# Stage 5 — Model C: LSTM with Attention

## Why LSTM + Attention?

Plain LSTMs can struggle to "look back" at a specific earlier time step  
when generating long forecasts. Attention solves this by learning a  
**weighted sum over all encoder hidden states** at each decoder step.

Architecture:
```
Encoder LSTM → hidden states H = [h_1, h_2, … h_L]
Attention     → context vector c_t = softmax(score(h_L, H)) · H
Decoder step  → LSTM(y_{t-1}, c_t) → y_t
```

**When attention helps most:** When the signal has sudden spikes or  
outage onsets that are important to "remember" but not captured by  
the most recent hidden state. The attention head learns to look back at  
the most recent non-outage window.

**Tradeoff:** Adds O(L) computation per decoder step → slower than GRU,  
but often more accurate at longer horizons.


In [ ]:
class BahdanauAttention(nn.Module):
    """Additive (Bahdanau) attention over encoder outputs."""
    def __init__(self, hidden_size):
        super().__init__()
        self.W_q = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_k = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v   = nn.Linear(hidden_size, 1,           bias=False)

    def forward(self, query, keys):
        """
        query : (B, H)      — decoder hidden state (last layer)
        keys  : (B, L, H)   — encoder hidden states
        Returns context: (B, H), attn_weights: (B, L)
        """
        q = self.W_q(query).unsqueeze(1)       # (B, 1, H)
        k = self.W_k(keys)                      # (B, L, H)
        scores = self.v(torch.tanh(q + k)).squeeze(-1)  # (B, L)
        weights = torch.softmax(scores, dim=-1)          # (B, L)
        context = (weights.unsqueeze(-1) * keys).sum(1)  # (B, H)
        return context, weights


class LSTMAttentionForecaster(nn.Module):
    """
    LSTM Encoder + Bahdanau Attention + LSTM Decoder.
    MIMO output: produces H predictions in one forward pass.
    """
    def __init__(self, hidden_size=128, num_layers=2,
                 horizon=FORECAST_HORIZON, dropout=0.2):
        super().__init__()
        self.horizon     = horizon
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        self.encoder = nn.LSTM(1, hidden_size, num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0.0)
        self.attn    = BahdanauAttention(hidden_size)
        self.decoder = nn.LSTM(1 + hidden_size, hidden_size, num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0.0)
        self.out_proj = nn.Linear(hidden_size, 1)
        self.drop     = nn.Dropout(dropout)

    def forward(self, x):
        """x: (B, L, 1)"""
        B = x.size(0)
        enc_out, (h, c) = self.encoder(x)     # enc_out: (B, L, H)
        dec_input = x[:, -1:, :]              # (B, 1, 1)
        outputs   = []

        for _ in range(self.horizon):
            query   = h[-1]                                    # (B, H)
            context, _ = self.attn(query, enc_out)            # (B, H)
            ctx_exp = context.unsqueeze(1)                    # (B, 1, H)
            dec_in  = torch.cat([dec_input, ctx_exp], dim=-1) # (B, 1, 1+H)
            out, (h, c) = self.decoder(dec_in, (h, c))
            pred    = self.out_proj(self.drop(out))           # (B, 1, 1)
            outputs.append(pred.squeeze(-1))                  # (B, 1)
            dec_input = pred

        return torch.cat(outputs, dim=1)   # (B, H)


lstm_attn = LSTMAttentionForecaster(hidden_size=128, num_layers=2,
                                     horizon=FORECAST_HORIZON, dropout=0.2)
total_params = sum(p.numel() for p in lstm_attn.parameters() if p.requires_grad)
print(f"LSTM+Attention — Trainable parameters: {total_params:,}")


In [ ]:
print("Training LSTM + Attention...")
lstm_train_losses, lstm_val_losses = train_model(
    lstm_attn, train_dl, val_dl,
    n_epochs   = 50,
    lr         = 5e-4,
    patience   = 10,
    model_name = "LSTM+Attention",
)
plot_loss_curve(lstm_train_losses, lstm_val_losses, "LSTM + Attention")


In [ ]:
lstm_preds  = dl_recursive_forecast(lstm_attn, train_norm, n_steps=FORECAST_HORIZON)
lstm_result = evaluate_dl(y_test_raw, lstm_preds, "LSTM+Attention (MIMO 3-hr)")
dl_leaderboard.append(lstm_result)
plot_dl_forecast(y_test_raw, lstm_preds, "LSTM + Attention")


---
# Stage 6 — Model D: Lightweight Transformer (PatchTST-style)

## Transformer for Time-Series — Caveats First

Vanilla Transformers (like the original NBeats/Informer) suffer from:  
1. **Quadratic attention** — O(L²) memory/compute, prohibitive at L=720  
2. **Point-wise tokens** — treating each 5-second step as a token loses local structure  
3. **Positional encoding** — sinusoidal PE designed for NLP, sub-optimal for TS

**Our approach: Patched Transformer (PatchTST-inspired)**  
- Divide the input window into non-overlapping **patches** of size `P`  
  → L/P tokens instead of L tokens → quadratic cost drops by P²  
- Each patch is projected to a d-dimensional embedding  
- Linear positional embedding (learnable) instead of sinusoidal  
- Efficient enough for L=720, P=12 → 60 tokens → fast attention

This reflects the state-of-the-art approach (Nie et al., PatchTST, 2023)  
adapted for production constraints.


In [ ]:
class PatchEmbedding(nn.Module):
    """Divide time series into patches and project to embedding dimension."""
    def __init__(self, patch_size, d_model, lookback):
        super().__init__()
        self.patch_size = patch_size
        n_patches = lookback // patch_size
        self.proj = nn.Linear(patch_size, d_model)
        self.pos  = nn.Embedding(n_patches, d_model)
        self.n_patches = n_patches

    def forward(self, x):
        """x: (B, L, 1) → (B, n_patches, d_model)"""
        B, L, _ = x.shape
        x = x.squeeze(-1)                           # (B, L)
        # Truncate to multiple of patch_size
        x = x[:, :self.n_patches * self.patch_size]
        x = x.reshape(B, self.n_patches, self.patch_size)  # (B, n_patches, P)
        x = self.proj(x)                            # (B, n_patches, d_model)
        pos = torch.arange(self.n_patches, device=x.device)
        x   = x + self.pos(pos).unsqueeze(0)        # add positional embedding
        return x


class PatchTransformerForecaster(nn.Module):
    """
    Lightweight Patched Transformer for MIMO multi-step forecasting.

    Pipeline:
      Patch Embedding → Transformer Encoder (N layers) →
      Flatten → Linear head → H outputs
    """
    def __init__(self, lookback, horizon, patch_size=12,
                 d_model=64, nhead=4, num_layers=3, dropout=0.1):
        super().__init__()
        assert lookback % patch_size == 0, "lookback must be divisible by patch_size"

        self.patch_embed = PatchEmbedding(patch_size, d_model, lookback)
        n_patches = lookback // patch_size

        encoder_layer = nn.TransformerEncoderLayer(
            d_model    = d_model,
            nhead      = nhead,
            dim_feedforward = d_model * 4,
            dropout    = dropout,
            activation = "gelu",
            batch_first= True,
            norm_first = True,   # Pre-LN for training stability
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.head = nn.Sequential(
            nn.Flatten(),                           # (B, n_patches * d_model)
            nn.Linear(n_patches * d_model, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, horizon),
        )

    def forward(self, x):
        """x: (B, L, 1)"""
        x = self.patch_embed(x)      # (B, n_patches, d_model)
        x = self.transformer(x)      # (B, n_patches, d_model)
        return self.head(x)          # (B, H)


PATCH_SIZE = 12   # each patch = 12 steps = 60 seconds

patch_transformer = PatchTransformerForecaster(
    lookback    = LOOKBACK,
    horizon     = FORECAST_HORIZON,
    patch_size  = PATCH_SIZE,
    d_model     = 64,
    nhead       = 4,
    num_layers  = 3,
    dropout     = 0.1,
)
patch_transformer = patch_transformer.to(DEVICE)

total_params = sum(p.numel() for p in patch_transformer.parameters() if p.requires_grad)
n_tokens = LOOKBACK // PATCH_SIZE
print(f"PatchTransformer — Trainable parameters : {total_params:,}")
print(f"Attention operates on {n_tokens} patch tokens (patch_size={PATCH_SIZE})")
print(f"Quadratic attention cost: {n_tokens}² = {n_tokens**2} (vs {LOOKBACK**2:,} for point-wise)")


In [ ]:
print("Training PatchTransformer...")
pt_train_losses, pt_val_losses = train_model(
    patch_transformer, train_dl, val_dl,
    n_epochs   = 50,
    lr         = 5e-4,
    patience   = 10,
    model_name = "PatchTransformer",
)
plot_loss_curve(pt_train_losses, pt_val_losses, "PatchTransformer")


In [ ]:
pt_preds  = dl_recursive_forecast(patch_transformer, train_norm, n_steps=FORECAST_HORIZON)
pt_result = evaluate_dl(y_test_raw, pt_preds, "PatchTransformer (MIMO 3-hr)")
dl_leaderboard.append(pt_result)
plot_dl_forecast(y_test_raw, pt_preds, "PatchTransformer")


---
# Stage 7 — Model E: N-BEATS (Optional — Interpretable DL)

## What is N-BEATS?

N-BEATS (Oreshkin et al., 2019) is a fully connected DL model designed  
*specifically* for univariate time-series forecasting with interpretability.

**Key innovations:**  
- **Doubly residual stacking**: each block produces a *backcast* (residual  
  explaining the input) and a *forecast* (contribution to the output)  
- **Basis expansion**: trend blocks use polynomial bases; seasonality blocks  
  use Fourier bases — creating *interpretable decompositions*  
- No recurrence, no convolutions → fully parallelisable, fast on CPU  
- No domain-specific features required — learns everything from the series alone

**Suitability here:**  
- ✅ Univariate sensor signal — exactly N-BEATS's design target  
- ✅ Interpretable trend/seasonality split is useful for IoT anomaly reporting  
- ⚠️ Generic (not interpretable) stack often outperforms on non-smooth series  
- ⚠️ Large horizon (2160) means wide FC output layers → memory intensive


In [ ]:
class NBEATSBlock(nn.Module):
    """
    Single N-BEATS block: shared FC stack → backcast + forecast basis weights.
    Generic stack (no explicit basis expansion) for robustness on noisy IoT signals.
    """
    def __init__(self, lookback, horizon, hidden_size=256, n_layers=4):
        super().__init__()
        self.lookback = lookback
        self.horizon  = horizon

        # Shared FC stack
        layers = []
        in_dim = lookback
        for _ in range(n_layers):
            layers += [nn.Linear(in_dim, hidden_size), nn.ReLU()]
            in_dim  = hidden_size
        self.fc_stack = nn.Sequential(*layers)

        # Backcast and forecast heads
        self.backcast_head = nn.Linear(hidden_size, lookback)
        self.forecast_head = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        """x: (B, L) flat"""
        h        = self.fc_stack(x)
        backcast = self.backcast_head(h)   # (B, L)
        forecast = self.forecast_head(h)   # (B, H)
        return backcast, forecast


class NBEATSStack(nn.Module):
    """Stack of N-BEATS blocks with doubly residual connections."""
    def __init__(self, lookback, horizon, n_blocks=4, hidden_size=256, n_layers=4):
        super().__init__()
        self.blocks = nn.ModuleList([
            NBEATSBlock(lookback, horizon, hidden_size, n_layers)
            for _ in range(n_blocks)
        ])

    def forward(self, x):
        """x: (B, L, 1) → forecast: (B, H)"""
        x = x.squeeze(-1)          # (B, L)
        forecast_sum = torch.zeros(x.size(0), self.blocks[0].horizon, device=x.device)

        for block in self.blocks:
            backcast, forecast = block(x)
            x            = x - backcast     # residual: subtract what this block explained
            forecast_sum = forecast_sum + forecast   # accumulate contributions

        return forecast_sum


nbeats = NBEATSStack(
    lookback     = LOOKBACK,
    horizon      = FORECAST_HORIZON,
    n_blocks     = 4,
    hidden_size  = 256,
    n_layers     = 4,
)
nbeats = nbeats.to(DEVICE)

total_params = sum(p.numel() for p in nbeats.parameters() if p.requires_grad)
print(f"N-BEATS — Trainable parameters: {total_params:,}")


In [ ]:
print("Training N-BEATS...")
nbeats_train_losses, nbeats_val_losses = train_model(
    nbeats, train_dl, val_dl,
    n_epochs   = 50,
    lr         = 5e-4,
    patience   = 10,
    model_name = "N-BEATS",
)
plot_loss_curve(nbeats_train_losses, nbeats_val_losses, "N-BEATS")


In [ ]:
nbeats_preds  = dl_recursive_forecast(nbeats, train_norm, n_steps=FORECAST_HORIZON)
nbeats_result = evaluate_dl(y_test_raw, nbeats_preds, "N-BEATS (MIMO 3-hr)")
dl_leaderboard.append(nbeats_result)
plot_dl_forecast(y_test_raw, nbeats_preds, "N-BEATS")


---
# Stage 8 — DL Leaderboard & Cross-Architecture Comparison


In [ ]:
# ── DL Leaderboard ────────────────────────────────────────────────────────────
dl_lb = (
    pd.DataFrame(dl_leaderboard)
    .sort_values("SMAPE")
    .reset_index(drop=True)
)
dl_lb.index += 1

print("\n===== DEEP LEARNING LEADERBOARD (sorted by SMAPE) =====")
print(dl_lb.to_string())


In [ ]:
# ── Visual leaderboard ────────────────────────────────────────────────────────
metrics = ["MAE", "RMSE", "SMAPE"]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric in zip(axes, metrics):
    sub = dl_lb[dl_lb[metric].notna()].copy()
    colors = ["#2ecc71" if i == 0 else "#3498db" for i in range(len(sub))]
    ax.barh(sub["Model"], sub[metric], color=colors, edgecolor="white")
    ax.set_title(metric, fontsize=13)
    ax.invert_yaxis()
    ax.set_xlabel(metric)

plt.suptitle("DL Model Leaderboard — Lower is Better", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── All DL models overlaid on the same test window ───────────────────────────
n_show = min(FORECAST_HORIZON, len(y_test_raw))
t_ax   = np.arange(n_show)

fig, ax = plt.subplots(figsize=(16, 7))
ax.plot(t_ax, y_test_raw[:n_show],      label="Actual",          linewidth=1.0, color="black",     alpha=0.85)
ax.plot(t_ax, tcn_preds[:n_show],       label="TCN",             linewidth=0.9, color="tomato",    alpha=0.85)
ax.plot(t_ax, gru_preds[:n_show],       label="GRU Seq2Seq",     linewidth=0.9, color="steelblue", alpha=0.8)
ax.plot(t_ax, lstm_preds[:n_show],      label="LSTM + Attention", linewidth=0.9, color="seagreen",  alpha=0.8)
ax.plot(t_ax, pt_preds[:n_show],        label="PatchTransformer",linewidth=0.9, color="purple",    alpha=0.75)
ax.plot(t_ax, nbeats_preds[:n_show],    label="N-BEATS",         linewidth=0.9, color="goldenrod", alpha=0.75)

ax.set_title("3-Hour Forecast — All DL Models vs Actual", fontsize=13)
ax.set_xlabel("Step (× 5 seconds)")
ax.set_ylabel("Reading")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()


In [ ]:
# ── Horizon-sliced SMAPE decay ────────────────────────────────────────────────
horizon_bins = [60, 180, 360, 720, 1080, 1440, 2160]
results_h = []

for name, preds in [
    ("TCN",             tcn_preds),
    ("GRU Seq2Seq",     gru_preds),
    ("LSTM+Attention",  lstm_preds),
    ("PatchTransformer",pt_preds),
    ("N-BEATS",         nbeats_preds),
]:
    for h in horizon_bins:
        h_clip = min(h, len(y_test_raw), len(preds))
        yt = y_test_raw[:h_clip]
        yp = preds[:h_clip]
        denom = (np.abs(yt) + np.abs(yp)) / 2
        m = denom > 0
        s = float(np.mean(np.abs(yt[m]-yp[m])/denom[m])*100) if m.any() else np.nan
        results_h.append({"Model": name, "Horizon (steps)": h, "SMAPE": s})

horizon_df = pd.DataFrame(results_h)

fig, ax = plt.subplots(figsize=(12, 5))
for model_name, grp in horizon_df.groupby("Model"):
    ax.plot(grp["Horizon (steps)"], grp["SMAPE"], marker="o", label=model_name)

ax.set_title("SMAPE vs Forecast Horizon — DL Models", fontsize=13)
ax.set_xlabel("Horizon (steps)")
ax.set_ylabel("SMAPE (%)")
ax.legend()
plt.tight_layout()
plt.show()


---
# Stage 9 — DL vs ML Combined Comparison

Cross-notebook summary. Populate `ml_results` below with values from  
Notebook 1's leaderboard (copy-paste the SMAPE / MAE values).


In [ ]:
# ── Combined leaderboard (manual entry from Notebook 1 results) ──────────────
# Edit these values to match your actual Notebook 1 output:
ml_results = [
    {"Model": "XGBoost (Recursive 3-hr)",      "MAE": None, "RMSE": None, "SMAPE": None, "Track": "ML"},
    {"Model": "LightGBM (Recursive 3-hr)",     "MAE": None, "RMSE": None, "SMAPE": None, "Track": "ML"},
    {"Model": "RandomForest (Recursive 3-hr)", "MAE": None, "RMSE": None, "SMAPE": None, "Track": "ML"},
]

dl_results = [{**r, "Track": "DL"} for r in dl_leaderboard]
combined   = pd.DataFrame(ml_results + dl_results).sort_values("SMAPE").reset_index(drop=True)
combined.index += 1

print("\n===== COMBINED ML vs DL LEADERBOARD =====")
print(combined.to_string())


In [ ]:
# ── Architecture decision matrix ──────────────────────────────────────────────
decision_matrix = {
    "Architecture":       ["TCN",     "GRU Seq2Seq",   "LSTM+Attn",  "PatchTransformer", "N-BEATS",   "XGBoost",   "LightGBM"],
    "Train Speed":        ["Fast",    "Medium",        "Slow",       "Medium",            "Fast",      "Fast",      "Fastest"],
    "Inference Speed":    ["Fast",    "Medium",        "Medium",     "Fast",              "Fast",      "Fast",      "Fastest"],
    "Memory Footprint":   ["Low",     "Medium",        "Medium",     "Low",               "Medium",    "Low",       "Low"],
    "Long Horizon":       ["Good",    "Good",          "Good",       "Good",              "Good",      "Degrades",  "Degrades"],
    "Outage Handling":    ["Implicit","Via gate",       "Via gate",   "Implicit",          "Implicit",  "Explicit",  "Explicit"],
    "Interpretability":   ["Low",     "Low",           "Attn maps",  "Low",               "High",      "SHAP",      "SHAP"],
    "GPU Required":       ["No",      "Preferred",     "Preferred",  "No",                "No",        "No",        "No"],
}
pd.set_option("display.max_colwidth", 20)
print(pd.DataFrame(decision_matrix).to_string(index=False))


---
# Stage 10 — Production Deployment Strategy

## Recommended Architecture: TCN + LightGBM Ensemble

For this specific problem (5-second IoT telemetry, 3-hour horizon):

```
PRIMARY:   LightGBM (Recursive) + Outage Mask
                → Best balance of accuracy, speed, interpretability
                → Can run on CPU in production without GPU dependency

SECONDARY: TCN (MIMO)
                → Use as a real-time sanity check / ensemble member
                → Parallelisable inference, deterministic, no hidden state

ENSEMBLE:  0.6 × LightGBM + 0.4 × TCN
                → Typically reduces SMAPE by 5-15% over either alone
                → Simple average or learned stacking on a holdout set
```

## Model Serving Architecture

```python
class ProductionForecaster:
    def __init__(self, lgb_model, tcn_model, regime_clf,
                 feature_cols, mu, sigma, lookback, horizon):
        self.lgb    = lgb_model
        self.tcn    = tcn_model
        self.regime = regime_clf
        ...

    def predict(self, recent_readings: np.ndarray,
                start_ts: pd.Timestamp) -> dict:
        # 1. Detect regime (outage vs normal)
        regime = self.regime.predict(self._regime_features(recent_readings))

        # 2. LightGBM recursive forecast
        lgb_preds = self._lgb_recursive(recent_readings, start_ts)

        # 3. TCN MIMO forecast
        tcn_preds = self._tcn_mimo(recent_readings)

        # 4. Ensemble
        ensemble  = 0.6 * lgb_preds + 0.4 * tcn_preds

        # 5. Outage override
        ensemble[regime == 1] = 0.0

        return {
            "point_forecast": ensemble,
            "lgb_forecast":   lgb_preds,
            "tcn_forecast":   tcn_preds,
        }
```

## Monitoring Checklist

```
[ ] Log SMAPE on rolling 24-hr holdout — alert if > 2× baseline
[ ] Monitor lag_1 correlation in production data vs training
[ ] Track zero_run_length distribution — outage pattern shift = retrain trigger
[ ] Weekly full retraining (both LightGBM and TCN)
[ ] Store all predictions with timestamps for drift analysis
[ ] A/B test ensemble weights quarterly
```

## Computational Budget (per 3-hour forecast, ~2160 steps)

| Component | CPU Time (est.) | GPU Time (est.) |
|---|---|---|
| LightGBM feature build | ~50ms | N/A |
| LightGBM recursive (2160 steps) | ~2s | N/A |
| TCN MIMO (single forward pass) | ~30ms | ~5ms |
| Regime classifier | ~5ms | N/A |
| **Total** | **~2.1s** | **~60ms (GPU)** |

Well within a real-time 5-second update cycle on any modern server.
